# ATDL Phase 5 — Imbalance-Aware Training Methods

Build four comparable training datasets from the Phase 4 moderate variant:
E1 ordinary imbalanced training; E2 repeat-factor oversampling; E3 targeted geometric/brightness augmentation; E4 combined oversampling and augmentation. Transforms are applied to training images only; all boxes are kept and transformed with the image. Validation/test stay in the Phase 2 folders. Training is opt-in.

## 1. Paths, settings, and prerequisites

In [1]:
from pathlib import Path
import hashlib, json, math, shutil, random
from collections import Counter
import pandas as pd
import yaml
from PIL import Image, ImageEnhance

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "requirements-local.txt").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the atdl_final_project folder in VS Code.")
DATA_ROOT = PROJECT_ROOT / "data"
VARIANTS_ROOT = DATA_ROOT / "imbalance_variants"
BASE_LEVEL = "moderate"
BASE_ROOT = VARIANTS_ROOT / BASE_LEVEL
BASE_MANIFEST = BASE_ROOT / "selection_manifest.csv"
METHODS_ROOT = VARIANTS_ROOT / "methods"
RESULTS = PROJECT_ROOT / "results"
CLASS_NAMES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
SEED = 42
REPEAT_CAP = 3
MINORITY_MAX_RATIO = 0.80
FORCE_REBUILD_METHODS = False
RUN_TRAINING = True
MODEL_WEIGHTS = "yolov8n.pt"
EPOCHS, IMAGE_SIZE, BATCH_SIZE, PATIENCE = 50, 640, 16, 12

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not BASE_MANIFEST.is_file():
    raise FileNotFoundError("Run Phase 4 first to create the moderate training variant.")
base = pd.read_csv(BASE_MANIFEST)
print(f"Base variant: {BASE_LEVEL}, {len(base)} training images.")


Base variant: moderate, 808 training images.


## 2. Read labels and identify minority classes

In [2]:
def read_label(path):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f"Invalid label line in {path}: {line}")
        cid = int(fields[0])
        if cid not in range(len(CLASS_NAMES)):
            raise ValueError(f"Invalid class ID in {path}: {cid}")
        rows.append((cid, [float(v) for v in fields[1:]]))
    return rows

image_counts, object_counts = Counter(), Counter()
records = []
for row in base.itertuples(index=False):
    labels = read_label(row.label)
    present = sorted({cid for cid, _ in labels})
    image_counts.update(present)
    object_counts.update(cid for cid, _ in labels)
    records.append({"image": Path(row.image), "label": Path(row.label), "classes": present, "labels": labels})
max_count = max(image_counts.values())
minority_classes = {cid for cid, count in image_counts.items() if count < MINORITY_MAX_RATIO * max_count}
display(pd.DataFrame([{
    "class_name": CLASS_NAMES[cid], "training_images": image_counts[cid],
    "objects": object_counts[cid], "minority": cid in minority_classes,
} for cid in range(len(CLASS_NAMES))]))
if not minority_classes:
    print("No class falls below the configured minority threshold; targeted augmentation will be empty.")


,class_name,training_images,objects,minority
0,crazing,121,275,True
1,inclusion,267,687,False
2,patches,214,553,False
3,pitted_surface,161,233,True
4,rolled-in_scale,81,172,True
5,scratches,54,103,True


## 3. Deterministic box-aware augmentation helpers

In [3]:
def label_text(rows):
    return "".join(str(cid) + " " + " ".join(f"{v:.8f}" for v in coords) + "\n" for cid, coords in rows)

def horizontal_flip(rows):
    return [(cid, [1.0 - xc, yc, bw, bh]) for cid, (xc, yc, bw, bh) in rows]

def rotate_ccw_90(rows):
    # NEU-DET images are square; normalized center and size transform with the image.
    return [(cid, [yc, 1.0 - xc, bh, bw]) for cid, (xc, yc, bw, bh) in rows]

def stable_suffix(path, operation):
    return hashlib.sha1((Path(path).name + operation).encode("utf-8")).hexdigest()[:8]

def write_image_and_labels(source_image, rows, destination_root, stem, operation):
    out_image = destination_root / "images" / "train" / f"{stem}.jpg"
    out_label = destination_root / "labels" / "train" / f"{stem}.txt"
    out_image.parent.mkdir(parents=True, exist_ok=True)
    out_label.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(source_image) as image:
        if operation == "original":
            transformed = image.convert("RGB")
        elif operation == "horizontal_flip":
            transformed = image.convert("RGB").transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        elif operation == "rotate_ccw_90":
            transformed = image.convert("RGB").transpose(Image.Transpose.ROTATE_90)
        elif operation == "brightness":
            transformed = ImageEnhance.Brightness(image.convert("RGB")).enhance(1.06)
        else:
            raise ValueError(operation)
        transformed.save(out_image, quality=95)
    transformed_rows = rows
    if operation == "horizontal_flip":
        transformed_rows = horizontal_flip(rows)
    elif operation == "rotate_ccw_90":
        transformed_rows = rotate_ccw_90(rows)
    out_label.write_text(label_text(transformed_rows), encoding="utf-8")
    return out_image, out_label


## 4. Build E1–E4 datasets without touching validation or test

In [4]:
EXTRA_STANDARD_VARIANTS = {
    "E1_mild_standard": VARIANTS_ROOT / "mild",
    "E1_severe_standard": VARIANTS_ROOT / "severe",
}
METHOD_SPECS = {
    "E1_imbalanced_standard": "original",
    "E2_repeat_factor": "repeat",
    "E3_targeted_augmentation": "augment",
    "E4_combined": "combined",
}
METHODS_ROOT.mkdir(parents=True, exist_ok=True)
validation_dir = DATA_ROOT / "processed" / "images" / "val"
test_dir = DATA_ROOT / "processed" / "images" / "test"
names_yaml = {i: name for i, name in enumerate(CLASS_NAMES)}
max_image_count = max(image_counts.values())
repeat_factor = {
    cid: min(REPEAT_CAP, max(1, math.ceil(math.sqrt(max_image_count / max(count, 1)))))
    for cid, count in image_counts.items()
}
print("Repeat factors:", {CLASS_NAMES[cid]: repeat_factor[cid] for cid in sorted(repeat_factor)})
method_summary_rows = []

for method_name, method in METHOD_SPECS.items():
    method_root = METHODS_ROOT / method_name
    manifest_path = method_root / "method_manifest.csv"
    reusable = manifest_path.is_file()
    if reusable:
        previous = pd.read_csv(manifest_path)
        reusable = len(previous) > 0 and all(Path(p).is_file() for p in previous["image"]) and all(
            Path(p).is_file() for p in previous["label"]
        )
    if reusable and not FORCE_REBUILD_METHODS:
        method_frame = pd.read_csv(manifest_path)
        print(f"Reusing completed {method_name} ({len(method_frame)} images).")
    else:
        if method_root.exists() and any(method_root.iterdir()) and not FORCE_REBUILD_METHODS:
            raise FileExistsError(f"Partial method folder at {method_root}; inspect it or set FORCE_REBUILD_METHODS=True.")
        rows_out = []
        for record in records:
            source_stem = record["image"].stem
            copies = 1
            if method in {"repeat", "combined"}:
                copies = max(repeat_factor.get(cid, 1) for cid in record["classes"])
            for repeat_index in range(copies):
                suffix = "" if repeat_index == 0 else f"_rep{repeat_index}"
                stem = f"{source_stem}{suffix}"
                out_image, out_label = write_image_and_labels(
                    record["image"], record["labels"], method_root, stem, "original"
                )
                rows_out.append({
                    "image": str(out_image), "label": str(out_label),
                    "source_image": str(record["image"]), "source_label": str(record["label"]),
                    "operation": "original", "classes_present": ",".join(CLASS_NAMES[c] for c in record["classes"]),
                })
            if method in {"augment", "combined"} and minority_classes.intersection(record["classes"]):
                for operation in ("horizontal_flip", "rotate_ccw_90", "brightness"):
                    aug_stem = f"{source_stem}_{operation}_{stable_suffix(record['image'], operation)}"
                    out_image, out_label = write_image_and_labels(
                        record["image"], record["labels"], method_root, aug_stem, operation
                    )
                    rows_out.append({
                        "image": str(out_image), "label": str(out_label),
                        "source_image": str(record["image"]), "source_label": str(record["label"]),
                        "operation": operation, "classes_present": ",".join(CLASS_NAMES[c] for c in record["classes"]),
                    })
        method_frame = pd.DataFrame(rows_out)
        method_frame.to_csv(manifest_path, index=False)

    yaml_data = {
        "path": str(method_root.resolve()), "train": "images/train",
        "val": str(validation_dir.resolve()), "test": str(test_dir.resolve()),
        "names": names_yaml,
    }
    (method_root / "data.yaml").write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding="utf-8")
    object_count, image_presence_count = Counter(), Counter()
    for label_path in method_frame["label"]:
        labels = read_label(label_path)
        object_count.update(cid for cid, _ in labels)
        image_presence_count.update({cid for cid, _ in labels})
    for cid, class_name in enumerate(CLASS_NAMES):
        method_summary_rows.append({
            "experiment": method_name, "class_name": class_name,
            "train_images_containing_class": image_presence_count[cid],
            "train_objects": object_count[cid], "total_train_images": len(method_frame),
        })

method_distribution = pd.DataFrame(method_summary_rows)
method_distribution.to_csv(RESULTS / "phase5_method_distribution.csv", index=False)
display(method_distribution)


Repeat factors: {'crazing': 2, 'inclusion': 1, 'patches': 2, 'pitted_surface': 2, 'rolled-in_scale': 2, 'scratches': 3}
Reusing completed E1_imbalanced_standard (808 images).
Reusing completed E2_repeat_factor (1459 images).
Reusing completed E3_targeted_augmentation (2056 images).
Reusing completed E4_combined (2707 images).


,experiment,class_name,train_images_containing_class,train_objects,total_train_images
0,E1_imbalanced_standard,crazing,121,275,808
1,E1_imbalanced_standard,inclusion,267,687,808
2,E1_imbalanced_standard,patches,214,553,808
3,E1_imbalanced_standard,pitted_surface,161,233,808
4,E1_imbalanced_standard,rolled-in_scale,81,172,808
5,E1_imbalanced_standard,scratches,54,103,808
6,E2_repeat_factor,crazing,242,550,1459
7,E2_repeat_factor,inclusion,353,814,1459
8,E2_repeat_factor,patches,429,1107,1459
9,E2_repeat_factor,pitted_surface,323,467,1459


## 5. Validate boxes, source membership, and unchanged fixed splits

In [5]:
def image_names(folder):
    return sorted(p.name for p in Path(folder).glob("*") if p.is_file())

val_before = image_names(validation_dir)
test_before = image_names(test_dir)
base_sources = {str(Path(p).resolve()) for p in base["image"]}
for method_name in METHOD_SPECS:
    method_root = METHODS_ROOT / method_name
    method_frame = pd.read_csv(method_root / "method_manifest.csv")
    for row in method_frame.itertuples(index=False):
        if str(Path(row.source_image).resolve()) not in base_sources:
            raise ValueError(f"Non-Phase-4 source image in {method_name}: {row.source_image}")
        for cid, (xc, yc, bw, bh) in read_label(row.label):
            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                raise ValueError(f"Bad transformed box in {row.label}")
            if xc-bw/2 < -1e-6 or xc+bw/2 > 1+1e-6 or yc-bh/2 < -1e-6 or yc+bh/2 > 1+1e-6:
                raise ValueError(f"Transformed box outside image in {row.label}")
    print(method_name, "images:", len(method_frame), "label rows validated.")
if image_names(validation_dir) != val_before or image_names(test_dir) != test_before:
    raise RuntimeError("Validation/test image directories changed unexpectedly.")
print("Validation and test image directories are unchanged.")


E1_imbalanced_standard images: 808 label rows validated.
E2_repeat_factor images: 1459 label rows validated.
E3_targeted_augmentation images: 2056 label rows validated.
E4_combined images: 2707 label rows validated.
Validation and test image directories are unchanged.


## 6. Optional matched training runs (validation only)

In [6]:
RUN_TRAINING = True
FORCE_RETRAIN = False
EPOCHS, IMAGE_SIZE, BATCH_SIZE, SEED = 50, 640, 16, 42
try:
    import torch
    import ultralytics
    from ultralytics import YOLO
    runtime_ready = True
    print("PyTorch", torch.__version__, "| Ultralytics", ultralytics.__version__,
          "| CUDA", torch.cuda.is_available())
except ImportError as exc:
    runtime_ready = False
    print("Training dependencies unavailable:", exc)

if RUN_TRAINING:
    if not runtime_ready:
        raise RuntimeError("Install CUDA-compatible PyTorch and Ultralytics in the project .venv first.")
    validation_metrics = []
    experiment_inputs = {}
    for method_name in METHOD_SPECS:
        method_root = METHODS_ROOT / method_name
        experiment_inputs[method_name] = {
            "data_yaml": method_root / "data.yaml",
            "signature_inputs": [method_root / "method_manifest.csv", method_root / "data.yaml",
                                 BASE_MANIFEST, DATA_ROOT / "splits" / "phase2_image_split.csv"],
        }
    for experiment_name, variant_root in EXTRA_STANDARD_VARIANTS.items():
        experiment_inputs[experiment_name] = {
            "data_yaml": variant_root / "data.yaml",
            "signature_inputs": [variant_root / "selection_manifest.csv", variant_root / "data.yaml",
                                 DATA_ROOT / "splits" / "phase2_image_split.csv"],
        }

    run_root = PROJECT_ROOT / "outputs" / "checkpoints" / "phase5"
    for experiment_name, spec in experiment_inputs.items():
        data_yaml_path = spec["data_yaml"]
        if not data_yaml_path.is_file():
            raise FileNotFoundError(f"Missing dataset YAML for {experiment_name}: {data_yaml_path}")
        run_path = run_root / experiment_name
        best = run_path / "weights" / "best.pt"
        signature_path = run_path / "run_signature.json"
        signature_inputs = spec["signature_inputs"]
        run_signature = {
            "input_sha256": {str(path.name): sha256_file(path) for path in signature_inputs},
            "model": MODEL_WEIGHTS,
            "model_weights_sha256": sha256_file(PROJECT_ROOT / MODEL_WEIGHTS)
                if (PROJECT_ROOT / MODEL_WEIGHTS).is_file() else None,
            "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH_SIZE,
            "seed": SEED, "patience": 12,
        }
        signature_matches = False
        if best.is_file() and signature_path.is_file():
            signature_matches = json.loads(signature_path.read_text(encoding="utf-8")) == run_signature
        if best.is_file() and signature_matches and not FORCE_RETRAIN:
            print("Checkpoint matches current data and settings:", best)
        else:
            if run_path.exists():
                from datetime import datetime
                stale_root = PROJECT_ROOT / "archive" / "stale_runs" / "phase5"
                stale_root.mkdir(parents=True, exist_ok=True)
                stale_path = stale_root / f"{datetime.now().strftime('%Y%m%dT%H%M%S_%f')}_{experiment_name}"
                shutil.move(str(run_path), str(stale_path))
                print("Archived stale/forced-retrain run:", stale_path)
            YOLO(MODEL_WEIGHTS).train(
                data=str(data_yaml_path), epochs=EPOCHS, imgsz=IMAGE_SIZE,
                batch=BATCH_SIZE, seed=SEED, patience=12, project=str(run_root),
                name=experiment_name, exist_ok=True,
                device=0 if torch.cuda.is_available() else "cpu", workers=4, deterministic=True,
            )
            if best.is_file():
                signature_path.write_text(json.dumps(run_signature, indent=2) + "\n", encoding="utf-8")
        if best.is_file():
            val_result = YOLO(str(best)).val(
                data=str(data_yaml_path), split="val", imgsz=IMAGE_SIZE,
                project=str(PROJECT_ROOT / "outputs" / "logs" / "phase5"),
                name=experiment_name, exist_ok=True, device=0 if torch.cuda.is_available() else "cpu",
            )
            row = {
                "experiment": experiment_name, "checkpoint_sha256": sha256_file(best),
                **{str(k): float(v) for k, v in getattr(val_result, "results_dict", {}).items()},
            }
            box = val_result.box
            for position, class_id in enumerate(list(getattr(box, "ap_class_index", []))):
                class_id = int(class_id)
                class_name = val_result.names[class_id]
                row[f"Precision_{class_name}"] = float(box.p[position]) if len(box.p) > position else None
                row[f"Recall_{class_name}"] = float(box.r[position]) if len(box.r) > position else None
                row[f"AP50_{class_name}"] = float(box.ap50[position]) if len(box.ap50) > position else None
                row[f"AP50_95_{class_name}"] = float(box.ap[position].mean()) if len(box.ap) > position else None
                precision, recall = row[f"Precision_{class_name}"], row[f"Recall_{class_name}"]
                row[f"F1_{class_name}"] = (2 * precision * recall / (precision + recall)) if precision is not None and recall is not None and precision + recall else 0.0
            row["macro_F1"] = sum(row.get(f"F1_{name}", 0.0) for name in CLASS_NAMES) / len(CLASS_NAMES)
            global_precision = row.get("metrics/precision(B)")
            global_recall = row.get("metrics/recall(B)")
            row["metrics/F1(B)"] = (2 * global_precision * global_recall / (global_precision + global_recall)) if global_precision is not None and global_recall is not None and global_precision + global_recall else 0.0
            validation_metrics.append(row)
    if validation_metrics:
        pd.DataFrame(validation_metrics).to_csv(RESULTS / "phase5_validation_comparison.csv", index=False)
        display(pd.DataFrame(validation_metrics).set_index("experiment"))
else:
    print("Training skipped; set RUN_TRAINING=True after installing the training runtime.")


PyTorch 2.15.0+cu132 | Ultralytics 8.4.215 | CUDA True
Checkpoint matches current data and settings: ./outputs/checkpoints/phase5/E1_imbalanced_standard/weights/best.pt


Ultralytics 8.4.215 🚀 Python-3.14.9 torch-2.15.0+cu132 CUDA:0 (NVIDIA RTX 4000 Ada Generation, 20019MiB)


Model summary (fused): 73 layers, 3,133,922 parameters, 0 gradients, 8.6 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2394.8±818.8 MB/s, size: 17.8 KB)


val: Scanning ./data/processed/labels/val.cache... 266 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 266/266 101.4Mit/s 0.0s

val: ./data/processed/images/val/crazing_120_3a37b2b9.jpg: 1 duplicate labels removed


... (2,818 outputs left out)

             scratches         45         78      0.632      0.679      0.747      0.417


Speed: 0.4ms preprocess, 1.6ms inference, 0.0ms loss, 0.6ms postprocess per image


Results saved to ./outputs/logs/phase5/E1_severe_standard


,checkpoint_sha256,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),fitness,Precision_crazing,Recall_crazing,AP50_crazing,AP50_95_crazing,...,AP50_rolled-in_scale,AP50_95_rolled-in_scale,F1_rolled-in_scale,Precision_scratches,Recall_scratches,AP50_scratches,AP50_95_scratches,F1_scratches,macro_F1,metrics/F1(B)
experiment,,,,,,,,,,,,,,,,,,,,,
E1_imbalanced_standard,08c95c14d836e1d2d8842016c3881959175965c76a8c90...,0.622227,0.643525,0.685249,0.386046,0.386046,0.398395,0.220764,0.297749,0.113482,...,0.466350,0.240699,0.398378,0.607606,0.833333,0.815871,0.469704,0.702790,0.623987,0.632697
E2_repeat_factor,95a4041f8dda8c6e49ddf16bee1aeb7f6c03c08dd866e8...,0.638675,0.659159,0.685498,0.382035,0.382035,0.414587,0.182130,0.283768,0.103029,...,0.532435,0.260792,0.543053,0.638287,0.948718,0.864419,0.519080,0.763141,0.633706,0.648755
E3_targeted_augmentation,d1ad3ebadc8404ea43e765ad129724f0614e61bdc7f37e...,0.640208,0.660560,0.687216,0.383106,0.383106,0.442951,0.371429,0.352285,0.121418,...,0.461326,0.234265,0.457414,0.688490,0.858974,0.845197,0.493588,0.764341,0.645536,0.650225
E4_combined,3008aa687b3780b88ca2118a2c4514845b26db41bedecb...,0.689798,0.650909,0.689874,0.374052,0.374052,0.486858,0.380952,0.366389,0.115552,...,0.466970,0.183924,0.458230,0.771893,0.867700,0.818711,0.485854,0.816997,0.666669,0.669790
E1_mild_standard,a1c12c61e6a62fde8ee2feca1e355044fb600b2bca5811...,0.668144,0.719607,0.741969,0.419188,0.419188,0.554087,0.355054,0.428636,0.166422,...,0.552431,0.272591,0.545934,0.702238,0.923077,0.908927,0.533254,0.797654,0.685287,0.692921
E1_severe_standard,dc0881b9edafd33a80f89c59eebae02599eff1ab52d6c6...,0.676544,0.555233,0.664679,0.372081,0.372081,0.419505,0.009524,0.252253,0.097662,...,0.479093,0.214931,0.459510,0.631875,0.679487,0.746948,0.416562,0.654817,0.575036,0.609915


## 7. Save reproducibility record

In [7]:
phase5_summary = {
    "base_variant": BASE_LEVEL, "seed": SEED,
    "training_enabled": RUN_TRAINING,
    "training_runtime_available": runtime_ready,
    "training_settings": {"model": MODEL_WEIGHTS, "epochs": EPOCHS, "imgsz": IMAGE_SIZE,
                          "batch": BATCH_SIZE, "patience": PATIENCE},
    "minority_classes": [CLASS_NAMES[cid] for cid in sorted(minority_classes)],
    "repeat_factor_by_class": {CLASS_NAMES[cid]: repeat_factor[cid] for cid in repeat_factor},
    "validation_test_unchanged": True,
    "experiments": {
        **{name: {
            "manifest": str((METHODS_ROOT / name / "method_manifest.csv").resolve()),
            "dataset_yaml": str((METHODS_ROOT / name / "data.yaml").resolve()),
            "train_images": int(len(pd.read_csv(METHODS_ROOT / name / "method_manifest.csv"))),
            "training_type": "imbalance-aware method on moderate variant",
        } for name in METHOD_SPECS},
        **{name: {
            "manifest": str((variant_root / "selection_manifest.csv").resolve()),
            "dataset_yaml": str((variant_root / "data.yaml").resolve()),
            "train_images": int(len(pd.read_csv(variant_root / "selection_manifest.csv"))),
            "training_type": "standard training at controlled imbalance severity",
        } for name, variant_root in EXTRA_STANDARD_VARIANTS.items()},
    },
    "distribution_report": str((RESULTS / "phase5_method_distribution.csv").resolve()),
}
(RESULTS / "phase5_summary.json").write_text(json.dumps(phase5_summary, indent=2) + "\n", encoding="utf-8")
print("Saved Phase 5 summary:", RESULTS / "phase5_summary.json")


Saved Phase 5 summary: ./results/phase5_summary.json


Report per-class precision, recall, AP and class frequencies. Do not select the method from overall mAP alone; validation is for model selection, and the test split is reserved for Phase 6.